# One practical tip on how to talk to your AI, with receipts — reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alestainer/statistics-intuitions/blob/main/notebooks/05-pronouns-thinking-tokens.ipynb)

This notebook contains the exact 16 matched prompt pairs, the complete resumable OpenRouter runner, and the saved measurements used in the article. Paid calls are opt-in and protected by an aggregate budget.


In [ ]:
from pathlib import Path
import json, urllib.request

RAW = "https://raw.githubusercontent.com/Alestainer/statistics-intuitions/main/"

def load_json(relative_path):
    for path in (Path("../") / relative_path, Path(relative_path)):
        if path.exists():
            return json.loads(path.read_text())
    with urllib.request.urlopen(RAW + relative_path) as response:
        return json.load(response)

benchmark = load_json("data/05-pronouns-thinking-tokens/benchmark.json")
results = load_json("data/05-pronouns-thinking-tokens/results.json")
print(f"{len(benchmark['items'])} prompt pairs; {len(results['per_model'])} models; {results['calls']} calls")


## Inspect one matched pair


In [ ]:
item = benchmark["items"][0]
print("EXPLICIT NAMES\n")
print(item["prompts"]["explicit"])
print("\n\nPRONOUNS / REFERENCES\n")
print(item["prompts"]["referential"])
print("\nExpected:", item["answer"])


## Run the full experiment yourself

These cells reproduce the API experiment, not just the analysis. They send all 16 matched pairs to the selected models, preserve each request and raw response, record provider, latency, usage and cost, and resume from completed calls.

Nothing paid runs by default. To run it, add `OPENROUTER_API_KEY` to Colab Secrets, set `RUN_PAID_EVAL = True`, and choose a budget. The original 192 calls cost $0.1301, but prices and routing can change. The conservative ceiling printed below assumes every response consumes the full 2,048-token completion allowance, so it is much higher than the observed cost.


In [ ]:
RUN_PAID_EVAL = False
BUDGET_USD = 4.00
RERUN_DIR = Path("pronoun-reference-rerun")  # Use a Google Drive path to persist across runtimes.

MODEL_SPECS = {
    "openai/gpt-5.6-sol": {"input": 2.0, "output": 10.0, "require_parameters": False},
    "google/gemini-3.8-flash": {"input": 0.75, "output": 3.75, "require_parameters": True},
    "anthropic/claude-opus-5": {"input": 5.0, "output": 25.0, "require_parameters": False},
    "anthropic/claude-sonnet-5": {"input": 2.0, "output": 10.0, "require_parameters": False},
    "qwen/qwen3.8-max": {"input": 2.0, "output": 6.0, "require_parameters": False},
    "z-ai/glm-5.3-flash": {"input": 0.075, "output": 0.25, "require_parameters": False},
}
MODELS_TO_RUN = list(MODEL_SPECS)  # Replace with one model id for a cheaper validation run.
MAX_TOKENS = 2048
SEED = 20260903
ENDPOINT = 'https://openrouter.ai/api/v1/chat/completions'


In [ ]:
import hashlib, os, re, time, urllib.error
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def json_sha256(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(encoded.encode()).hexdigest()

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n")

def append_jsonl(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a") as handle:
        handle.write(json.dumps(value, ensure_ascii=False) + "\n")

def schedule(manifest):
    calls = []
    for item in manifest["items"]:
        conditions = ("explicit", "referential") if int(item["id"]) % 2 else ("referential", "explicit")
        for within_pair_order, condition in enumerate(conditions, 1):
            calls.append({
                "call_id": f"item-{item['id']}--{condition}",
                "item": item,
                "condition": condition,
                "within_pair_order": within_pair_order,
                "scheduled_order": len(calls) + 1,
            })
    return calls

def request_payload(model, spec, call):
    return {
        "model": model,
        "messages": [
            {"role": "system", "content": benchmark["system_prompt"]},
            {"role": "user", "content": call["item"]["prompts"][call["condition"]]},
        ],
        "temperature": 0,
        "seed": SEED,
        "max_tokens": MAX_TOKENS,
        "usage": {"include": True},
        "reasoning": {"effort": "low", "exclude": True},
        "provider": {
            "sort": "price",
            "allow_fallbacks": False,
            "require_parameters": spec["require_parameters"],
            "max_price": {"prompt": spec["input"], "completion": spec["output"]},
        },
    }

def worst_case_cost(payload, spec):
    # UTF-8 bytes are a conservative upper bound on input tokens; add chat framing.
    input_ceiling = len(json.dumps(payload["messages"], ensure_ascii=False).encode()) + 256
    return input_ceiling * spec["input"] / 1_000_000 + MAX_TOKENS * spec["output"] / 1_000_000

def send(payload, api_key, timeout=180):
    request = urllib.request.Request(
        ENDPOINT,
        data=json.dumps(payload).encode(),
        method="POST",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://alestainer.com",
            "X-OpenRouter-Title": "pronoun reference-style reasoning replication",
        },
    )
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            status, body = response.status, response.read().decode()
    except urllib.error.HTTPError as error:
        status, body = error.code, error.read().decode(errors="replace")
    try:
        response_json = json.loads(body)
    except json.JSONDecodeError:
        response_json = {"unparsed_body": body}
    return status, response_json, time.perf_counter() - started

def response_content(response):
    try:
        content = response["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError):
        return None
    return content if isinstance(content, str) else None

def parse_answer(content):
    if not isinstance(content, str):
        return None, False
    normalized = content.strip().upper()
    if re.fullmatch(r"OPTION [ABCD]", normalized):
        return normalized, True
    final_line = normalized.splitlines()[-1].strip()
    if re.fullmatch(r"OPTION [ABCD]", final_line):
        return final_line, False
    labels = re.findall(r"\bOPTION [ABCD]\b", normalized)
    return (labels[0] if len(set(labels)) == 1 else None), False

def usage_fields(response):
    usage = response.get("usage") if isinstance(response.get("usage"), dict) else {}
    details = usage.get("completion_tokens_details")
    details = details if isinstance(details, dict) else {}
    reasoning = details.get("reasoning_tokens")
    cost = usage.get("cost")
    return {
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "total_tokens": usage.get("total_tokens"),
        "reasoning_tokens": reasoning if isinstance(reasoning, (int, float)) else None,
        "cost_usd": cost if isinstance(cost, (int, float)) else None,
        "raw_usage": usage,
    }

MANIFEST_HASH = json_sha256(benchmark)
CALLS = schedule(benchmark)
PLANNED = [
    (model, MODEL_SPECS[model], call, request_payload(model, MODEL_SPECS[model], call))
    for model in MODELS_TO_RUN for call in CALLS
]
CONSERVATIVE_MAX_USD = sum(worst_case_cost(payload, spec) for _, spec, _, payload in PLANNED)
print(f"Planned calls: {len(PLANNED)}")
print(f"Frozen manifest SHA-256: {MANIFEST_HASH}")
print(f"Conservative full-run ceiling: ${CONSERVATIVE_MAX_USD:.4f}")


In [ ]:
def result_paths():
    return sorted(RERUN_DIR.glob("models/*/calls/*/result.json"))

def load_existing():
    existing = {}
    for path in result_paths():
        row = json.loads(path.read_text())
        if row.get("manifest_sha256") != MANIFEST_HASH:
            raise RuntimeError(f"Manifest mismatch in {path}")
        existing[(row["model"], row["call_id"])] = row
    return existing

def get_api_key():
    key = os.environ.get("OPENROUTER_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("OPENROUTER_API_KEY")
        except Exception:
            key = None
    if not key:
        import getpass
        key = getpass.getpass("OpenRouter API key: ")
    return key

def run_paid_eval():
    if BUDGET_USD <= 0 or BUDGET_USD > 5:
        raise ValueError("BUDGET_USD must be above $0 and no more than the notebook hard cap of $5")
    if BUDGET_USD + 1e-12 < CONSERVATIVE_MAX_USD:
        print(f"Budget ${BUDGET_USD:.4f} is below the ${CONSERVATIVE_MAX_USD:.4f} conservative full-run ceiling; the runner may stop early.")

    api_key = get_api_key()
    RERUN_DIR.mkdir(parents=True, exist_ok=True)
    config_path = RERUN_DIR / "config.json"
    config = {
        "created_at": utc_now(), "endpoint": ENDPOINT, "models": MODELS_TO_RUN,
        "model_specs": MODEL_SPECS, "temperature": 0, "reasoning_effort": "low",
        "reasoning_excluded": True, "seed": SEED, "max_tokens": MAX_TOKENS,
        "manifest_sha256": MANIFEST_HASH, "approved_aggregate_budget_usd": BUDGET_USD,
        "conservative_full_suite_cost_usd": CONSERVATIVE_MAX_USD,
    }
    if config_path.exists():
        prior = json.loads(config_path.read_text())
        for key in ("models", "model_specs", "seed", "max_tokens", "manifest_sha256"):
            if prior.get(key) != config.get(key):
                raise RuntimeError(f"Resume configuration mismatch: {key}")
        if BUDGET_USD > float(prior["approved_aggregate_budget_usd"]):
            raise RuntimeError("A resume cannot raise the original budget; use a new output directory")
    else:
        write_json(config_path, config)

    existing = load_existing()
    if any(row.get("cost_usd") is None for row in existing.values()):
        raise RuntimeError("Cannot resume safely: a completed call is missing reported cost")
    spent = sum(float(row["cost_usd"]) for row in existing.values())

    for global_order, (model, spec, call, payload) in enumerate(PLANNED, 1):
        key = (model, call["call_id"])
        if key in existing:
            continue
        reservation = worst_case_cost(payload, spec)
        if spent + reservation > BUDGET_USD + 1e-12:
            print(f"STOP budget: spent ${spent:.6f}; next call reserves ${reservation:.6f}; cap ${BUDGET_USD:.6f}")
            break

        safe_model = model.replace("/", "--")
        call_dir = RERUN_DIR / "models" / safe_model / "calls" / call["call_id"]
        write_json(call_dir / "request.json", payload)
        print(f"run {global_order:03d}/{len(PLANNED)} {model}/{call['call_id']}")
        requested_at = utc_now()
        status, response, latency = send(payload, api_key)
        completed_at = utc_now()
        write_json(call_dir / "raw-response.json", response)
        append_jsonl(RERUN_DIR / "raw-responses.jsonl", {
            "model": model, "call_id": call["call_id"], "http_status": status, "response": response,
        })
        if status < 200 or status >= 300 or "error" in response:
            write_json(call_dir / "error.json", {
                "model": model, "call_id": call["call_id"], "http_status": status,
                "requested_at": requested_at, "completed_at": completed_at,
                "latency_seconds": latency, "response": response,
            })
            print(f"STOP API error: HTTP {status}. Run the cell again to resume.")
            break

        usage = usage_fields(response)
        content = response_content(response)
        parsed, strictly_formatted = parse_answer(content)
        item = call["item"]
        result = {
            "call_id": call["call_id"], "manifest_sha256": MANIFEST_HASH,
            "item_id": item["id"], "condition": call["condition"],
            "analysis_set": item["analysis_set"], "ambiguity": item["ambiguity"],
            "family": item["family"], "near_length_matched": item["near_length_matched"],
            "entity_count": item["entity_count"],
            "max_antecedent_distance_words_approx": item["max_antecedent_distance_words_approx"],
            "expected_answer": item["answer"], "parsed_answer": parsed,
            "strictly_formatted": strictly_formatted, "correct": parsed == item["answer"],
            "raw_content": content, "scheduled_order_within_model": call["scheduled_order"],
            "scheduled_order_global": global_order, "within_pair_order": call["within_pair_order"],
            "model": model, "returned_model": response.get("model"), "provider": response.get("provider"),
            "requested_at": requested_at, "completed_at": completed_at,
            "http_status": status, "latency_seconds": latency, **usage,
        }
        write_json(call_dir / "result.json", result)
        existing[key] = result
        if usage["cost_usd"] is None:
            print("STOP accounting: no reported cost. Run the cell again to resume past the saved call.")
            break
        spent += float(usage["cost_usd"])
        if usage["reasoning_tokens"] is None:
            print("STOP accounting: no reasoning-token count. It remains missing, not zero.")
            break
        if parsed is None:
            print("WARNING: no unique answer label; raw response retained and the run continues")

        other = "referential" if call["condition"] == "explicit" else "explicit"
        other_row = existing.get((model, f"item-{item['id']}--{other}"))
        if other_row and other_row.get("provider") != result.get("provider"):
            print(f"STOP provider changed within pair {item['id']}. Run the cell again to resume.")
            break

    summary = {
        "updated_at": utc_now(), "completed_calls": len(existing), "expected_calls": len(PLANNED),
        "reported_cost_usd": sum(float(row["cost_usd"]) for row in existing.values() if row.get("cost_usd") is not None),
        "missing_cost_calls": sum(row.get("cost_usd") is None for row in existing.values()),
        "missing_reasoning_token_calls": sum(row.get("reasoning_tokens") is None for row in existing.values()),
    }
    write_json(RERUN_DIR / "summary.json", summary)
    return summary

if RUN_PAID_EVAL:
    print(json.dumps(run_paid_eval(), indent=2))
else:
    print("DRY RUN ONLY — set RUN_PAID_EVAL = True to send the API requests")


## Analyze a completed or partial rerun


In [ ]:
rerun_files = result_paths()
if not rerun_files:
    print("No rerun results yet. The remaining sections verify the frozen published run.")
else:
    import pandas as pd
    rerun = pd.DataFrame(json.loads(path.read_text()) for path in rerun_files)
    usable = rerun[(rerun["analysis_set"] == "primary") & rerun["reasoning_tokens"].notna()]
    paired_rerun = usable.pivot(index=["model", "item_id"], columns="condition", values="reasoning_tokens").dropna()
    paired_rerun["delta"] = paired_rerun["referential"] - paired_rerun["explicit"]
    display(paired_rerun.groupby("model").agg(
        pairs=("delta", "size"), explicit_mean=("explicit", "mean"),
        referential_mean=("referential", "mean"), mean_paired_delta=("delta", "mean"),
    ))
    print("Rerun cost:", rerun["cost_usd"].dropna().sum())


## Recompute the primary table


In [ ]:
import pandas as pd

DISPLAY = {
    "openai/gpt-5.6-sol": "GPT-5.6 Sol",
    "google/gemini-3.8-flash": "Gemini 3.8 Flash",
    "anthropic/claude-opus-5": "Claude Opus 5",
    "anthropic/claude-sonnet-5": "Claude Sonnet 5",
    "qwen/qwen3.8-max": "Qwen 3.8 Max",
    "z-ai/glm-5.3-flash": "GLM 5.3 Flash",
}
ORDER = list(DISPLAY)
rows = []
for model in ORDER:
    primary = results["per_model"][model]["primary"]
    accuracy = primary["accuracy"]
    rows.append({
        "model": DISPLAY[model],
        "explicit mean": round(primary["explicit_reasoning_tokens"]["mean"], 1),
        "referential mean": round(primary["referential_reasoning_tokens"]["mean"], 1),
        "mean paired delta": round(primary["paired_delta_referential_minus_explicit"]["mean"], 1),
        "median paired delta": round(primary["paired_delta_referential_minus_explicit"]["median"], 1),
        "explicit correct": f'{round(accuracy["explicit"]["mean"] * 12)}/12',
        "referential correct": f'{round(accuracy["referential"]["mean"] * 12)}/12',
    })
pd.DataFrame(rows)


## Pooled paired result


In [ ]:
pairs = []
for model, model_result in results["per_model"].items():
    for pair in model_result["primary"]["pairs"]:
        pairs.append({"model": DISPLAY[model], **pair})

paired = pd.DataFrame(pairs)
print("Explicit mean:", round(paired["explicit_reasoning_tokens"].mean(), 1))
print("Referential mean:", round(paired["referential_reasoning_tokens"].mean(), 1))
print("Mean paired delta:", round(paired["delta_reasoning_tokens"].mean(), 1))
print("Median paired delta:", round(paired["delta_reasoning_tokens"].median(), 1))
print("Signs:", {
    "increased": int((paired["delta_reasoning_tokens"] > 0).sum()),
    "tied": int((paired["delta_reasoning_tokens"] == 0).sum()),
    "decreased": int((paired["delta_reasoning_tokens"] < 0).sum()),
})
print("Mean input-token delta:", round(paired["delta_prompt_tokens"].mean(), 1))


## Answer quality and ambiguity diagnostic


In [ ]:
def accuracy_totals(result_key):
    explicit = referential = total = 0
    for model_result in results["per_model"].values():
        for pair in model_result[result_key]["pairs"]:
            explicit += int(pair["explicit_correct"])
            referential += int(pair["referential_correct"])
            total += 1
    return explicit, referential, total

primary_accuracy = accuracy_totals("primary")
diagnostic_accuracy = accuracy_totals("ambiguous_diagnostic")
diagnostic_deltas = [
    pair["delta_reasoning_tokens"]
    for model_result in results["per_model"].values()
    for pair in model_result["ambiguous_diagnostic"]["pairs"]
]

print(f"Primary accuracy, explicit / referential: {primary_accuracy[0]}/{primary_accuracy[2]} / {primary_accuracy[1]}/{primary_accuracy[2]}")
print(f"Diagnostic accuracy, explicit / referential: {diagnostic_accuracy[0]}/{diagnostic_accuracy[2]} / {diagnostic_accuracy[1]}/{diagnostic_accuracy[2]}")
print("Diagnostic mean paired delta:", round(pd.Series(diagnostic_deltas).mean(), 1))
print("Diagnostic median paired delta:", round(pd.Series(diagnostic_deltas).median(), 1))


## Plot the model means


In [ ]:
import matplotlib.pyplot as plt

table = pd.DataFrame(rows).set_index("model")
ax = table[["explicit mean", "referential mean"]].plot.barh(figsize=(8, 4.5), color=["#94a3b8", "#7c3aed"])
ax.set_xlabel("Reported reasoning tokens, mean over 12 unambiguous questions")
ax.set_ylabel("")
ax.legend(["Explicit names", "Pronouns / references"])
plt.tight_layout()
plt.show()


## Scope

The four items labelled `diagnostic` are deliberately ambiguous and are excluded from the primary table. Reasoning-token zeroes are retained as reported; missing accounting would remain missing rather than being converted to zero.

The original requests used temperature 0, low reasoning effort and a 2,048-token completion cap. The rerun cells use those same settings, but models, providers, prices and provider-side implementations can change, so a new run is a replication rather than a deterministic replay.
